# Calibration, Uncertainty, and Error Analysis Lab


## Purpose

This lab examines calibration and grouped error diagnostics.

Aggregate performance can hide overconfidence and uneven failure patterns.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 2000

frame = pd.DataFrame({
    "group": rng.choice(["A", "B", "C"], size=n, p=[0.50, 0.30, 0.20]),
    "score": rng.beta(2.0, 3.0, size=n),
})

frame["target"] = rng.binomial(1, frame["score"])
frame["prediction"] = (frame["score"] >= 0.50).astype(int)
frame["error"] = frame["prediction"] != frame["target"]

frame["confidence_bin"] = pd.cut(
    frame["score"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

calibration = (
    frame
    .groupby("confidence_bin", observed=True)
    .agg(
        n=("target", "size"),
        mean_confidence=("score", "mean"),
        empirical_rate=("target", "mean"),
    )
    .reset_index()
)

grouped = (
    frame
    .groupby("group", as_index=False)
    .agg(
        sample_size=("error", "size"),
        classification_error_rate=("error", "mean"),
        base_rate=("target", "mean"),
        selection_rate=("prediction", "mean"),
    )
)

calibration, grouped

## Interpretation

Calibration asks whether probability scores mean what they claim. Grouped diagnostics ask whether model failures are evenly distributed or systematically concentrated.